# Experiment Pipeline — APTOS 2019 Diabetic Retinopathy
## Sequential Model Comparison: CViTS-Net vs Dual-Expert Attention

Tests individual techniques on both models under **identical conditions**.  
Baseline experiments remain **untouched** — all outputs go to `experiments/`.

**Techniques tested (independently):**
1. Rotation Augmentation
2. Class Weight Loss
3. CLAHE Preprocessing

**Output structure:**
```
experiments/
  cvits/
    rotation/ | class_weight/ | clahe/
  dual_expert/
    rotation/ | class_weight/ | clahe/
```
Each folder contains: `best_model.pth`, `training_log.csv`, `metrics.json`, `plots/`

## 1. Imports & Environment

In [2]:
import os
import sys
import warnings
import time
import json
warnings.filterwarnings('ignore')

sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from sklearn.metrics import auc as sklearn_auc

try:
    import cv2
    HAS_CV2 = True
except ImportError:
    HAS_CV2 = False
    print("\u26a0 OpenCV not installed. CLAHE experiment will be skipped.")
    print("  Install with: pip install opencv-python-headless")

from dataset_loader import get_data_loaders
from metrics import MetricsCalculator
from cvitsnet_model import build_cvitsnet
from dual_expert_model import build_dual_expert

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"OpenCV: {'Available' if HAS_CV2 else 'Not available'}")
print("\n\u2713 Environment ready")

PyTorch: 2.9.1+cu126
Device: cuda
GPU: NVIDIA RTX A5000
VRAM: 22.5 GB
OpenCV: Available

✓ Environment ready


## 2. Load Dataset (One-Time)

Load APTOS2019 data **once**. The same splits (`random_state=42`, stratified) are reused across all experiments.

In [3]:
print("Loading APTOS2019 Dataset...")
_, _, _, class_weights_dict, (X_train, y_train, X_val, y_val, X_test, y_test) = \
    get_data_loaders(dataset_path="APTOS2019", batch_size=16)

print(f"\nSplit sizes: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")
print(f"Class weights: {class_weights_dict}")

# Class distribution
unique, counts = np.unique(y_train, return_counts=True)
CLASS_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
print("\nTraining class distribution:")
for cls, name, count in zip(unique, CLASS_NAMES, counts):
    print(f"  Class {cls} ({name}): {count} samples ({count/len(y_train)*100:.1f}%)")

print("\n\u2713 Dataset loaded")

Loading APTOS2019 Dataset...
Total training samples: 3662
Class distribution:
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64
Loaded 100/3662 images
Loaded 200/3662 images
Loaded 300/3662 images
Loaded 400/3662 images
Loaded 500/3662 images
Loaded 600/3662 images
Loaded 700/3662 images
Loaded 800/3662 images
Loaded 900/3662 images
Loaded 1000/3662 images
Loaded 1100/3662 images
Loaded 1200/3662 images
Loaded 1300/3662 images
Loaded 1400/3662 images
Loaded 1500/3662 images
Loaded 1600/3662 images
Loaded 1700/3662 images
Loaded 1800/3662 images
Loaded 1900/3662 images
Loaded 2000/3662 images
Loaded 2100/3662 images
Loaded 2200/3662 images
Loaded 2300/3662 images
Loaded 2400/3662 images
Loaded 2500/3662 images
Loaded 2600/3662 images
Loaded 2700/3662 images
Loaded 2800/3662 images
Loaded 2900/3662 images
Loaded 3000/3662 images
Loaded 3100/3662 images
Loaded 3200/3662 images
Loaded 3300/3662 images
Loaded 3400/3662 images
Loaded 3500/3662 images
Loade

## 3. Experiment Configuration

All hyperparameters identical to the CViTS and Dual-Expert baseline experiments.  
Only the **technique** changes between experiments.

In [4]:
# ============================================================
# Base training config (identical to baselines)
# ============================================================
BASE_CONFIG = {
    'epochs': 100,
    'batch_size': 16,
    'lr': 0.001,
    'weight_decay': 0.0001,
    'num_classes': 5,
    'image_size': 224,
    'best_metric': 'val_qwk',       # monitor val_QWK, mode=max
    'checkpoint_interval': 10,       # periodic checkpoint every N epochs
}

# ============================================================
# Technique-specific experiment configs
# ============================================================
EXPERIMENTS = {
    'rotation': {
        'name': 'Rotation Augmentation',
        'augmentation': 'rotation',
        'preprocessing': None,
        'use_class_weights': False,
        'rotation_degrees': 15,
    },
    'class_weight': {
        'name': 'Class Weight Loss',
        'augmentation': None,
        'preprocessing': None,
        'use_class_weights': True,
    },
    'clahe': {
        'name': 'CLAHE Preprocessing',
        'augmentation': None,
        'preprocessing': 'clahe',
        'use_class_weights': False,
        'clahe_clip_limit': 2.0,
        'clahe_grid_size': [8, 8],
    },
}

MODELS = ['cvits', 'dual_expert']

print("Base Config:")
for k, v in BASE_CONFIG.items():
    print(f"  {k}: {v}")

print(f"\nExperiments ({len(EXPERIMENTS)}):")
for key, exp in EXPERIMENTS.items():
    print(f"  {key}: {exp['name']}")

print(f"\nModels: {MODELS}")
print(f"Total experiments: {len(EXPERIMENTS) * len(MODELS)}")

Base Config:
  epochs: 100
  batch_size: 16
  lr: 0.001
  weight_decay: 0.0001
  num_classes: 5
  image_size: 224
  best_metric: val_qwk
  checkpoint_interval: 10

Experiments (3):
  rotation: Rotation Augmentation
  class_weight: Class Weight Loss
  clahe: CLAHE Preprocessing

Models: ['cvits', 'dual_expert']
Total experiments: 6


## 4. Dataset Module

Custom dataset supporting:
- **Rotation augmentation** (training only, PIL-based)
- **CLAHE preprocessing** (all splits, OpenCV-based)
- **Baseline** (no changes, identical to existing loaders)

In [5]:
def apply_clahe(images, clip_limit=2.0, grid_size=(8, 8)):
    """Apply CLAHE preprocessing to a batch of uint8 images."""
    if not HAS_CV2:
        raise RuntimeError("OpenCV is required for CLAHE preprocessing.")
    processed = np.zeros_like(images)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)
    for i in range(len(images)):
        lab = cv2.cvtColor(images[i], cv2.COLOR_RGB2LAB)
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        processed[i] = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        if (i + 1) % 500 == 0:
            print(f"    CLAHE: {i+1}/{len(images)} images processed")
    return processed


class ExperimentDataset(Dataset):
    """Dataset with optional augmentation (rotation) support."""

    def __init__(self, images, labels, num_classes=5,
                 augmentation=None, rotation_degrees=15, is_training=True):
        self.images = images
        self.labels = labels
        self.num_classes = num_classes
        self.augmentation = augmentation
        self.rotation_degrees = rotation_degrees
        self.is_training = is_training

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]  # (224, 224, 3) uint8

        # Apply rotation augmentation (training only)
        if self.is_training and self.augmentation == 'rotation':
            pil_img = Image.fromarray(image)
            angle = np.random.uniform(-self.rotation_degrees, self.rotation_degrees)
            pil_img = pil_img.rotate(angle, resample=Image.BILINEAR, fillcolor=(0, 0, 0))
            image = np.array(pil_img)

        # Convert to tensor — same format as baseline: float32 [0-255]
        image_tensor = torch.from_numpy(image.copy()).permute(2, 0, 1).float()

        # One-hot label — same as baseline dataset loader
        label = np.zeros(self.num_classes, dtype=np.float32)
        label[self.labels[idx]] = 1.0
        label = torch.from_numpy(label)

        return image_tensor, label


def create_experiment_loaders(experiment_config, batch_size=16):
    """Create train/val/test DataLoaders for a specific experiment."""
    augmentation = experiment_config.get('augmentation')
    preprocessing = experiment_config.get('preprocessing')
    rotation_degrees = experiment_config.get('rotation_degrees', 15)

    train_imgs, val_imgs, test_imgs = X_train, X_val, X_test

    # Apply CLAHE preprocessing (to ALL splits — preprocessing, not augmentation)
    if preprocessing == 'clahe':
        clip = experiment_config.get('clahe_clip_limit', 2.0)
        grid = tuple(experiment_config.get('clahe_grid_size', [8, 8]))
        print("  Applying CLAHE preprocessing...")
        train_imgs = apply_clahe(X_train, clip, grid)
        val_imgs = apply_clahe(X_val, clip, grid)
        test_imgs = apply_clahe(X_test, clip, grid)
        print("  \u2713 CLAHE applied to all splits")

    # Create datasets
    train_ds = ExperimentDataset(
        train_imgs, y_train, augmentation=augmentation,
        rotation_degrees=rotation_degrees, is_training=True
    )
    val_ds = ExperimentDataset(val_imgs, y_val, is_training=False)
    test_ds = ExperimentDataset(test_imgs, y_test, is_training=False)

    # Create loaders (same settings as baseline)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             num_workers=0, pin_memory=True)

    return train_loader, val_loader, test_loader


print("\u2713 Dataset module ready")

✓ Dataset module ready


## 5. Model Factory

- **CViTS-Net**: imported from `cvitsnet_model.py`
- **Dual-Expert**: imported from `dual_expert_model.py`

In [6]:
def create_model(model_name, num_classes=5):
    """Factory function — creates a fresh model instance."""
    if model_name == 'cvits':
        model = build_cvitsnet(num_classes=num_classes, image_size=224)
    elif model_name == 'dual_expert':
        model = build_dual_expert(num_classes=num_classes)
    else:
        raise ValueError(f"Unknown model: {model_name}")

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Model: {model_name} | Parameters: {total_params:,}")
    return model


print("\u2713 Model factory ready")

✓ Model factory ready


## 6. Training Engine

Full training loop with:
- **Checkpoint resume** (auto-recovers from crashes)
- **Best model tracking** (val\_QWK, mode=max)
- **Per-epoch metrics logging** to CSV
- **Test evaluation** with best model checkpoint

In [7]:
def train_experiment(model, optimizer, train_loader, val_loader, test_loader,
                     experiment_dir, config, use_class_weights=False):
    """
    Train model with checkpoint resume capability.

    Returns:
        history, test_metrics, test_loss, cm, roc_data, best_epoch, best_val_qwk
    """
    experiment_dir = Path(experiment_dir)
    plots_dir = experiment_dir / 'plots'
    experiment_dir.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)

    epochs = config['epochs']
    num_classes = config['num_classes']

    # Class weight loss (optional)
    loss_weight = None
    if use_class_weights:
        weights = [class_weights_dict[i] for i in range(num_classes)]
        loss_weight = torch.FloatTensor(weights).to(device)
        print(f"  Class weights applied: {[f'{w:.3f}' for w in weights]}")

    # Initialize tracking
    metric_names = ['loss', 'accuracy', 'precision', 'recall',
                    'f1_score', 'specificity', 'roc_auc', 'qwk']
    history = {m: {'train': [], 'val': []} for m in metric_names}
    best_val_qwk = -1.0
    best_epoch = 0
    start_epoch = 0

    # ---- Resume from checkpoint if exists ----
    resume_path = experiment_dir / 'resume_checkpoint.pth'
    if resume_path.exists():
        print("  \u21bb Resuming from checkpoint...")
        ckpt = torch.load(str(resume_path), map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_val_qwk = ckpt['best_val_qwk']
        best_epoch = ckpt['best_epoch']
        history = ckpt['history']
        print(f"  \u21bb Resumed from epoch {start_epoch} (best QWK: {best_val_qwk:.4f})")

    # ---- Training loop ----
    for epoch in range(start_epoch, epochs):

        # ------ Train phase ------
        model.train()
        train_calc = MetricsCalculator(num_classes=num_classes)
        train_loss_sum, n_batches = 0.0, 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            preds = model(imgs)
            loss = F.cross_entropy(preds, labels, weight=loss_weight)
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item()
            train_calc.update(labels.cpu().numpy(),
                              F.softmax(preds, dim=1).detach().cpu().numpy())
            n_batches += 1

        train_loss = train_loss_sum / n_batches
        train_metrics = train_calc.compute_metrics()

        # ------ Validation phase ------
        model.eval()
        val_calc = MetricsCalculator(num_classes=num_classes)
        val_loss_sum, n_val = 0.0, 0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                preds = model(imgs)
                loss = F.cross_entropy(preds, labels, weight=loss_weight)
                val_loss_sum += loss.item()
                val_calc.update(labels.cpu().numpy(),
                                F.softmax(preds, dim=1).cpu().numpy())
                n_val += 1

        val_loss = val_loss_sum / n_val
        val_metrics = val_calc.compute_metrics()

        # ------ Store history ------
        history['loss']['train'].append(float(train_loss))
        history['loss']['val'].append(float(val_loss))
        for m in ['accuracy', 'precision', 'recall', 'f1_score',
                  'specificity', 'roc_auc', 'qwk']:
            if m in train_metrics:
                history[m]['train'].append(train_metrics[m])
                history[m]['val'].append(val_metrics[m])

        current_qwk = val_metrics.get('qwk', 0)

        # Print progress every 10 epochs + first epoch
        if (epoch + 1) % 10 == 0 or epoch == start_epoch:
            print(f"  Epoch {epoch+1:3d}/{epochs} | "
                  f"Loss: {train_loss:.4f}/{val_loss:.4f} | "
                  f"Acc: {train_metrics.get('accuracy',0):.4f}/{val_metrics.get('accuracy',0):.4f} | "
                  f"QWK: {train_metrics.get('qwk',0):.4f}/{current_qwk:.4f}")

        # ------ Best model (val_QWK max) ------
        if current_qwk > best_val_qwk:
            best_val_qwk = current_qwk
            best_epoch = epoch + 1
            torch.save({
                'epoch': best_epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_qwk': best_val_qwk,
                'val_loss': val_loss,
            }, str(experiment_dir / 'best_model.pth'))

        # ------ Resume checkpoint (every epoch) ------
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': None,
            'best_val_qwk': best_val_qwk,
            'best_epoch': best_epoch,
            'history': history,
        }, str(resume_path))

        # ------ Periodic checkpoint ------
        if (epoch + 1) % config.get('checkpoint_interval', 10) == 0:
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
            }, str(experiment_dir / f'checkpoint_epoch_{epoch+1:03d}.pth'))

    print(f"  \u2713 Training done. Best: Epoch {best_epoch} (QWK: {best_val_qwk:.4f})")

    # ---- Test evaluation with best model ----
    best_path = experiment_dir / 'best_model.pth'
    if best_path.exists():
        ckpt = torch.load(str(best_path), map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f"  Loaded best model (Epoch {ckpt['epoch']}, QWK: {ckpt['val_qwk']:.4f})")

    model.eval()
    test_calc = MetricsCalculator(num_classes=num_classes)
    test_loss_sum, n_test = 0.0, 0

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs)
            loss = F.cross_entropy(preds, labels)
            test_loss_sum += loss.item()
            test_calc.update(labels.cpu().numpy(),
                             F.softmax(preds, dim=1).cpu().numpy())
            n_test += 1

    test_loss = test_loss_sum / n_test
    test_metrics = test_calc.compute_metrics()
    cm = test_calc.get_confusion_matrix()
    roc_data = test_calc.get_roc_curves()

    print(f"  Test: Loss={test_loss:.4f} | Acc={test_metrics.get('accuracy',0):.4f} | "
          f"F1={test_metrics.get('f1_score',0):.4f} | QWK={test_metrics.get('qwk',0):.4f}")

    # ---- Save training log CSV ----
    log_data = {'epoch': list(range(1, len(history['loss']['train']) + 1))}
    for m in metric_names:
        log_data[f'train_{m}'] = history[m]['train']
        log_data[f'val_{m}'] = history[m]['val']
    pd.DataFrame(log_data).to_csv(str(experiment_dir / 'training_log.csv'), index=False)

    # ---- Save test metrics JSON ----
    def _convert(obj):
        if isinstance(obj, dict):
            return {k: _convert(v) for k, v in obj.items()}
        elif isinstance(obj, (list, tuple)):
            return [_convert(i) for i in obj]
        elif isinstance(obj, (np.integer, np.floating)):
            return obj.item()
        return obj

    metrics_output = {
        'best_epoch': best_epoch,
        'best_val_qwk': float(best_val_qwk),
        'test_loss': float(test_loss),
        **{f'test_{k}': float(v) for k, v in test_metrics.items()},
        'confusion_matrix': cm.tolist(),
    }
    with open(str(experiment_dir / 'metrics.json'), 'w') as f:
        json.dump(_convert(metrics_output), f, indent=4)

    # ---- Save full metrics CSV ----
    metrics_row = {
        'best_epoch': best_epoch,
        'best_val_qwk': float(best_val_qwk),
        'test_loss': float(test_loss),
    }
    for k, v in test_metrics.items():
        metrics_row[f'test_{k}'] = float(v)
    pd.DataFrame([metrics_row]).to_csv(str(experiment_dir / 'metrics.csv'), index=False)

    # Clean up resume checkpoint (training completed successfully)
    if resume_path.exists():
        os.remove(str(resume_path))

    return history, test_metrics, test_loss, cm, roc_data, best_epoch, best_val_qwk


print("\u2713 Training engine ready")

✓ Training engine ready


## 7. Visualization Module

Generates all required plots and saves as PNG (300 DPI).

In [8]:
def save_experiment_plots(history, cm, roc_data, experiment_dir, title_prefix):
    """Generate and save all visualization plots for an experiment."""
    plots_dir = Path(experiment_dir) / 'plots'
    plots_dir.mkdir(parents=True, exist_ok=True)
    epochs_range = range(1, len(history['loss']['train']) + 1)

    def _plot_curve(train_v, val_v, ylabel, title, fname, maximize=True):
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(epochs_range, train_v, 'b-', lw=2, label=f'Train')
        ax.plot(epochs_range, val_v, 'r-', lw=2, label=f'Validation')
        if val_v:
            best_idx = int(np.argmax(val_v)) if maximize else int(np.argmin(val_v))
            ax.scatter([best_idx+1], [val_v[best_idx]], c='green', s=100, zorder=5)
            ax.axvline(x=best_idx+1, color='green', ls='--', alpha=0.5,
                       label=f'Best: {val_v[best_idx]:.4f} (Ep {best_idx+1})')
        ax.set_xlabel('Epoch', fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(f'{title_prefix} \u2014 {title}', fontsize=14, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(str(plots_dir / fname), dpi=300, bbox_inches='tight')
        plt.close()

    # Metric curves
    _plot_curve(history['loss']['train'], history['loss']['val'],
               'Loss', 'Training vs Validation Loss', 'loss_curve.png', maximize=False)
    _plot_curve(history['accuracy']['train'], history['accuracy']['val'],
               'Accuracy', 'Accuracy Curve', 'accuracy_curve.png')
    _plot_curve(history['precision']['train'], history['precision']['val'],
               'Precision', 'Precision Curve', 'precision_curve.png')
    _plot_curve(history['recall']['train'], history['recall']['val'],
               'Recall', 'Recall Curve', 'recall_curve.png')
    _plot_curve(history['f1_score']['train'], history['f1_score']['val'],
               'F1-score', 'F1 Score Curve', 'f1_curve.png')
    _plot_curve(history['roc_auc']['train'], history['roc_auc']['val'],
               'ROC-AUC', 'ROC-AUC Curve', 'roc_auc_curve.png')
    _plot_curve(history['qwk']['train'], history['qwk']['val'],
               'QWK', 'QWK Curve', 'qwk_curve.png')

    # Confusion Matrix
    fig, ax = plt.subplots(figsize=(10, 8))
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_title(f'{title_prefix} \u2014 Confusion Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(plots_dir / 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
    plt.close()

    # ROC Curve
    if roc_data:
        fig, ax = plt.subplots(figsize=(10, 8))
        colors = plt.cm.Set1(np.linspace(0, 1, len(roc_data)))
        for i, (cls, vals) in enumerate(roc_data.items()):
            roc_auc_val = sklearn_auc(vals['fpr'], vals['tpr'])
            ax.plot(vals['fpr'], vals['tpr'], color=colors[i], lw=2,
                    label=f'{CLASS_NAMES[i]} (AUC={roc_auc_val:.3f})')
        ax.plot([0, 1], [0, 1], 'k--', lw=2)
        ax.set_xlabel('False Positive Rate', fontsize=12)
        ax.set_ylabel('True Positive Rate', fontsize=12)
        ax.set_title(f'{title_prefix} \u2014 ROC Curves', fontsize=14, fontweight='bold')
        ax.legend(loc='lower right', fontsize=10)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(str(plots_dir / 'roc_curve.png'), dpi=300, bbox_inches='tight')
        plt.close()

    # Class Distribution
    fig, ax = plt.subplots(figsize=(8, 5))
    unique, counts = np.unique(y_train, return_counts=True)
    bars = ax.bar(CLASS_NAMES, counts, color=plt.cm.Set2(np.linspace(0, 1, 5)))
    ax.set_ylabel('Sample Count', fontsize=12)
    ax.set_xlabel('DR Grade', fontsize=12)
    ax.set_title('APTOS2019 \u2014 Training Set Class Distribution', fontsize=14, fontweight='bold')
    for bar, n in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, n + 5, str(n),
                ha='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(plots_dir / 'class_distribution.png'), dpi=300, bbox_inches='tight')
    plt.close()

    print(f"  \u2713 Plots saved to {plots_dir}/")


print("\u2713 Visualization module ready")

✓ Visualization module ready


## 8. Experiment Runner

Orchestrates experiment execution:
- Skips already-completed experiments
- Saves cumulative results to `experiments/results_summary.csv`
- Cleans up GPU memory between runs

In [9]:
RESULTS_CSV = Path('experiments/results_summary.csv')


def run_experiment(model_name, technique, experiment_config=None):
    """
    Run a single experiment.

    Args:
        model_name: 'cvits' or 'dual_expert'
        technique: key from EXPERIMENTS dict
        experiment_config: override config (optional)

    Returns:
        dict with test results, or None if skipped
    """
    if experiment_config is None:
        experiment_config = EXPERIMENTS[technique]

    exp_name = f"{model_name}/{technique}"
    experiment_dir = Path('experiments') / model_name / technique

    print(f"\n{'='*80}")
    print(f"EXPERIMENT: {exp_name} \u2014 {experiment_config['name']}")
    print(f"{'='*80}")

    # Skip if already completed (metrics.json exists and no resume checkpoint)
    metrics_file = experiment_dir / 'metrics.json'
    resume_file = experiment_dir / 'resume_checkpoint.pth'
    if metrics_file.exists() and not resume_file.exists():
        print(f"  \u26a0 Already completed. Loading existing results.")
        with open(str(metrics_file)) as f:
            existing = json.load(f)
        print(f"  Test QWK: {existing.get('test_qwk', 'N/A')}")
        return existing

    # Check CLAHE availability
    if experiment_config.get('preprocessing') == 'clahe' and not HAS_CV2:
        print("  \u2717 Skipped: OpenCV not available for CLAHE")
        return None

    # 1. Create data loaders
    print("  Creating data loaders...")
    train_loader, val_loader, test_loader = create_experiment_loaders(
        experiment_config, batch_size=BASE_CONFIG['batch_size'])

    # 2. Create model
    print(f"  Building model: {model_name}")
    model = create_model(model_name, num_classes=BASE_CONFIG['num_classes'])
    model = model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=BASE_CONFIG['lr'],
        weight_decay=BASE_CONFIG['weight_decay']
    )

    # 3. Train
    print(f"  Training for {BASE_CONFIG['epochs']} epochs...")
    start_time = time.time()

    history, test_metrics, test_loss, cm, roc_data, best_epoch, best_qwk = \
        train_experiment(
            model, optimizer, train_loader, val_loader, test_loader,
            experiment_dir, BASE_CONFIG,
            use_class_weights=experiment_config.get('use_class_weights', False)
        )

    elapsed = time.time() - start_time
    print(f"  Training time: {elapsed/60:.1f} minutes")

    # 4. Generate plots
    print("  Generating visualizations...")
    save_experiment_plots(
        history, cm, roc_data, experiment_dir,
        f"{model_name.upper()} \u2014 {experiment_config['name']}"
    )

    # 5. Append to results CSV
    result_row = {
        'model': model_name,
        'technique': technique,
        'best_epoch': best_epoch,
        'best_val_qwk': float(best_qwk),
        'test_loss': float(test_loss),
        'training_time_min': round(elapsed / 60, 1),
    }
    for k, v in test_metrics.items():
        result_row[f'test_{k}'] = float(v)

    RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
    if RESULTS_CSV.exists():
        df = pd.read_csv(str(RESULTS_CSV))
        # Remove previous entry for same experiment if re-running
        df = df[~((df['model'] == model_name) & (df['technique'] == technique))]
        df = pd.concat([df, pd.DataFrame([result_row])], ignore_index=True)
    else:
        df = pd.DataFrame([result_row])
    df.to_csv(str(RESULTS_CSV), index=False)

    # 6. Cleanup GPU
    del model, optimizer
    torch.cuda.empty_cache()

    print(f"  \u2713 Experiment complete: {exp_name}")
    return result_row


def run_all_experiments(model_name):
    """Run all technique experiments for a specific model."""
    print(f"\n{'#'*80}")
    print(f"  RUNNING ALL EXPERIMENTS FOR: {model_name.upper()}")
    print(f"{'#'*80}")

    results = {}
    for technique in EXPERIMENTS:
        result = run_experiment(model_name, technique)
        if result:
            results[technique] = result

    print(f"\n\u2713 All {model_name} experiments complete ({len(results)}/{len(EXPERIMENTS)} succeeded)")
    return results


print("\u2713 Experiment runner ready")

✓ Experiment runner ready


## 9. Run a Single Experiment (Manual)

Uncomment and modify the cell below to run a **single** experiment manually.

In [10]:
# ============================================================
# Run a single experiment manually
# ============================================================
# Uncomment ONE of the following lines:

# result = run_experiment('cvits', 'rotation')
# result = run_experiment('cvits', 'class_weight')
# result = run_experiment('cvits', 'clahe')
# result = run_experiment('dual_expert', 'rotation')
# result = run_experiment('dual_expert', 'class_weight')
# result = run_experiment('dual_expert', 'clahe')

print("\u2191 Uncomment a line above to run a single experiment")

↑ Uncomment a line above to run a single experiment


## 10. Run All CViTS-Net Experiments

Execute all 3 technique experiments with CViTS-Net sequentially.

In [11]:
cvits_results = run_all_experiments('cvits')


################################################################################
  RUNNING ALL EXPERIMENTS FOR: CVITS
################################################################################

EXPERIMENT: cvits/rotation — Rotation Augmentation
  Creating data loaders...
  Building model: cvits
  Model: cvits | Parameters: 30,338,053
  Training for 100 epochs...
  Epoch   1/100 | Loss: 21.6027/3.4723 | Acc: 0.3597/0.5087 | QWK: 0.0790/0.2312
  Epoch  10/100 | Loss: 1.0146/0.8871 | Acc: 0.6453/0.6948 | QWK: 0.5614/0.6184
  Epoch  20/100 | Loss: 0.7886/0.7544 | Acc: 0.7248/0.7355 | QWK: 0.6876/0.6688
  Epoch  30/100 | Loss: 0.7868/0.7856 | Acc: 0.7186/0.7238 | QWK: 0.6822/0.6624
  Epoch  40/100 | Loss: 0.7708/0.8019 | Acc: 0.7165/0.7093 | QWK: 0.6980/0.5782
  Epoch  50/100 | Loss: 0.7884/0.8146 | Acc: 0.7190/0.7384 | QWK: 0.6924/0.6326
  Epoch  60/100 | Loss: 0.7553/0.7692 | Acc: 0.7273/0.7326 | QWK: 0.6861/0.7070
  Epoch  70/100 | Loss: 0.7760/0.8194 | Acc: 0.7136/0.6919 | QWK: 0

## 11. Run All Dual-Expert Experiments

Execute all 3 technique experiments with Dual-Expert Attention sequentially.

In [12]:
dual_expert_results = run_all_experiments('dual_expert')


################################################################################
  RUNNING ALL EXPERIMENTS FOR: DUAL_EXPERT
################################################################################

EXPERIMENT: dual_expert/rotation — Rotation Augmentation
  Creating data loaders...
  Building model: dual_expert
  Model: dual_expert | Parameters: 11,244,713
  Training for 100 epochs...
  Epoch   1/100 | Loss: 0.9475/0.8477 | Acc: 0.6586/0.6773 | QWK: 0.5222/0.4651
  Epoch  10/100 | Loss: 0.7650/0.7886 | Acc: 0.7211/0.7209 | QWK: 0.6773/0.6278
  Epoch  20/100 | Loss: 0.6632/0.6536 | Acc: 0.7565/0.7674 | QWK: 0.7546/0.6942
  Epoch  30/100 | Loss: 0.5899/0.6144 | Acc: 0.7735/0.7791 | QWK: 0.7938/0.7411
  Epoch  40/100 | Loss: 0.5402/0.6674 | Acc: 0.7960/0.7587 | QWK: 0.8397/0.7768
  Epoch  50/100 | Loss: 0.4339/0.6334 | Acc: 0.8397/0.7616 | QWK: 0.8773/0.8015
  Epoch  60/100 | Loss: 0.3595/0.7423 | Acc: 0.8634/0.7674 | QWK: 0.8955/0.8136
  Epoch  70/100 | Loss: 0.2469/0.9933 | Acc:

## 12. Results Comparison

Load all experiment results and display a comparative summary table.

In [13]:
print("\n" + "="*80)
print("EXPERIMENT RESULTS COMPARISON")
print("="*80)

if RESULTS_CSV.exists():
    df = pd.read_csv(str(RESULTS_CSV))

    # Display key metrics
    display_cols = ['model', 'technique', 'best_epoch',
                    'test_accuracy', 'test_precision', 'test_recall',
                    'test_f1_score', 'test_specificity', 'test_roc_auc',
                    'test_qwk', 'test_loss']
    cols = [c for c in display_cols if c in df.columns]

    print("\n" + df[cols].to_string(index=False))

    # Best technique per model
    print("\n" + "-"*40)
    for model_name in df['model'].unique():
        model_df = df[df['model'] == model_name]
        if 'test_qwk' in model_df.columns and not model_df['test_qwk'].isna().all():
            best = model_df.loc[model_df['test_qwk'].idxmax()]
            print(f"Best for {model_name}: {best['technique']} "
                  f"(QWK={best['test_qwk']:.4f}, Acc={best.get('test_accuracy', 0):.4f})")

    print(f"\n\u2713 Full results saved: {RESULTS_CSV}")
else:
    print("No results found. Run experiments first.")

print("\n" + "="*80)
print("\u2705 Pipeline complete!")
print("="*80)


EXPERIMENT RESULTS COMPARISON

      model    technique  best_epoch  test_accuracy  test_precision  test_recall  test_f1_score  test_specificity  test_roc_auc  test_qwk  test_loss
      cvits     rotation          18       0.698690        0.619409     0.698690       0.632899          0.912844      0.898227  0.689409   0.817241
      cvits class_weight          80       0.580786        0.708247     0.580786       0.608760          0.902555      0.897420  0.665551   0.994692
      cvits        clahe          37       0.696507        0.709940     0.696507       0.643582          0.913667      0.901563  0.684611   0.852905
dual_expert     rotation          80       0.759825        0.749929     0.759825       0.752744          0.936681      0.931912  0.817062   1.139636
dual_expert class_weight          44       0.699782        0.732798     0.699782       0.704336          0.926342      0.923662  0.753334   0.829845
dual_expert        clahe          43       0.739083        0.732711     0.

---

# Part 2 — Advanced Combined Experiments

## Additional Imbalance-Handling & Augmentation Techniques

Three new combined-technique experiments appended to the existing pipeline:

1. **Rotation + Focal Loss** — rotation augmentation with focal loss for class imbalance
2. **Rotation + MixUp** — rotation augmentation with MixUp regularization
3. **CLAHE + Rotation + Focal Loss** — full pipeline combining all three

These reuse the existing dataset loaders, model definitions, and visualization modules.

**Output structure:**
```
experiments/
  cvits/
    rotation_focal_loss/ | rotation_mixup/ | clahe_rotation_focal_loss/
  dual_expert/
    rotation_focal_loss/ | rotation_mixup/ | clahe_rotation_focal_loss/
```

## 13. Focal Loss & MixUp Modules

In [ ]:
# ============================================================
# Focal Loss — handles class imbalance by down-weighting easy examples
# Numerically stable using log_softmax
# ============================================================
class FocalLoss(nn.Module):
    """
    Focal Loss for multi-class classification with one-hot labels.
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    Uses log_softmax for numerical stability.
    """
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha  # Optional per-class weights (tensor)
        self.reduction = reduction

    def forward(self, inputs, targets):
        """
        Args:
            inputs: (B, C) raw logits
            targets: (B, C) one-hot encoded labels
        """
        # log_softmax is numerically stable (avoids softmax → log instability)
        log_probs = F.log_softmax(inputs, dim=1)
        targets_idx = targets.argmax(dim=1)

        # Gather log(p_t) and compute p_t from it
        log_p_t = log_probs.gather(1, targets_idx.unsqueeze(1)).squeeze(1)
        p_t = log_p_t.exp()

        # Focal weight: (1 - p_t)^gamma
        focal_weight = (1.0 - p_t) ** self.gamma

        # Apply alpha (per-class weights) if provided
        if self.alpha is not None:
            alpha_t = self.alpha.gather(0, targets_idx)
            focal_weight = alpha_t * focal_weight

        # loss = -alpha_t * (1-p_t)^gamma * log(p_t)
        loss = -focal_weight * log_p_t

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


# ============================================================
# MixUp — regularization by interpolating training pairs
# ============================================================
def mixup_data(x, y, alpha=0.2):
    """
    Apply MixUp augmentation to a batch.

    Args:
        x: (B, C, H, W) image tensor
        y: (B, num_classes) one-hot label tensor
        alpha: Beta distribution parameter

    Returns:
        mixed_x, mixed_y, lam
    """
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0

    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    mixed_y = lam * y + (1 - lam) * y[index]

    return mixed_x, mixed_y, lam


print("\u2713 Focal Loss (stable, gamma=2.0) and MixUp (alpha=0.2) modules ready")

✓ Focal Loss (gamma=2.0) and MixUp (alpha=0.2) modules ready


## 14. Advanced Experiment Configuration

Combined-technique experiments. All other hyperparameters remain identical to the baselines.

In [15]:
ADVANCED_EXPERIMENTS = {
    'rotation_focal_loss': {
        'name': 'Rotation + Focal Loss',
        'augmentation': 'rotation',
        'preprocessing': None,
        'use_class_weights': False,
        'rotation_degrees': 15,
        'loss': 'focal',
        'focal_gamma': 2.0,
        'mixup': False,
    },
    'rotation_mixup': {
        'name': 'Rotation + MixUp',
        'augmentation': 'rotation',
        'preprocessing': None,
        'use_class_weights': False,
        'rotation_degrees': 15,
        'loss': 'cross_entropy',
        'mixup': True,
        'mixup_alpha': 0.2,
    },
    'clahe_rotation_focal_loss': {
        'name': 'CLAHE + Rotation + Focal Loss',
        'augmentation': 'rotation',
        'preprocessing': 'clahe',
        'use_class_weights': False,
        'rotation_degrees': 15,
        'clahe_clip_limit': 2.0,
        'clahe_grid_size': [8, 8],
        'loss': 'focal',
        'focal_gamma': 2.0,
        'mixup': False,
    },
}

print(f"Advanced Experiments ({len(ADVANCED_EXPERIMENTS)}):")
for key, exp in ADVANCED_EXPERIMENTS.items():
    print(f"  {key}: {exp['name']}")
print(f"\nModels: {MODELS}")
print(f"Total new experiments: {len(ADVANCED_EXPERIMENTS) * len(MODELS)}")

Advanced Experiments (3):
  rotation_focal_loss: Rotation + Focal Loss
  rotation_mixup: Rotation + MixUp
  clahe_rotation_focal_loss: CLAHE + Rotation + Focal Loss

Models: ['cvits', 'dual_expert']
Total new experiments: 6


## 15. Advanced Training Engine

Extended training loop supporting:
- **Focal Loss** (gamma=2.0) as alternative to cross-entropy
- **MixUp** augmentation (alpha=0.2) applied in-batch during training
- All existing features: checkpoint resume, best model (val\_QWK), per-epoch CSV logging, test evaluation

In [ ]:
def train_advanced_experiment(model, optimizer, train_loader, val_loader, test_loader,
                              experiment_dir, config, experiment_config):
    """
    Advanced training loop with Focal Loss and MixUp support.

    Returns:
        history, test_metrics, test_loss, cm, roc_data, best_epoch, best_val_qwk
    """
    experiment_dir = Path(experiment_dir)
    plots_dir = experiment_dir / 'plots'
    experiment_dir.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)

    epochs = config['epochs']
    num_classes = config['num_classes']

    # ---- Loss function ----
    loss_type = experiment_config.get('loss', 'cross_entropy')
    use_mixup = experiment_config.get('mixup', False)
    mixup_alpha = experiment_config.get('mixup_alpha', 0.2)

    if loss_type == 'focal':
        gamma = experiment_config.get('focal_gamma', 2.0)
        criterion = FocalLoss(gamma=gamma)
        print(f"  Loss: Focal Loss (gamma={gamma})")
    else:
        criterion = None  # Will use F.cross_entropy
        print(f"  Loss: Cross-Entropy")

    if use_mixup:
        print(f"  MixUp: enabled (alpha={mixup_alpha})")

    # ---- Initialize tracking ----
    metric_names = ['loss', 'accuracy', 'precision', 'recall',
                    'f1_score', 'specificity', 'roc_auc', 'qwk']
    history = {m: {'train': [], 'val': []} for m in metric_names}
    best_val_qwk = -1.0
    best_epoch = 0
    start_epoch = 0

    # ---- Resume from checkpoint if exists ----
    resume_path = experiment_dir / 'resume_checkpoint.pth'
    if resume_path.exists():
        print("  \u21bb Resuming from checkpoint...")
        ckpt = torch.load(str(resume_path), map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_val_qwk = ckpt['best_val_qwk']
        best_epoch = ckpt['best_epoch']
        history = ckpt['history']
        print(f"  \u21bb Resumed from epoch {start_epoch} (best QWK: {best_val_qwk:.4f})")

    # ---- Training loop ----
    for epoch in range(start_epoch, epochs):

        # ------ Train phase ------
        model.train()
        train_calc = MetricsCalculator(num_classes=num_classes)
        train_loss_sum, n_batches = 0.0, 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            # Apply MixUp (training only)
            if use_mixup:
                imgs, labels_mixed, lam = mixup_data(imgs, labels, alpha=mixup_alpha)
            else:
                labels_mixed = labels

            optimizer.zero_grad()
            preds = model(imgs)

            # Compute loss
            if criterion is not None:
                loss = criterion(preds, labels_mixed)
            else:
                loss = F.cross_entropy(preds, labels_mixed)

            loss.backward()
            # Gradient clipping to prevent exploding gradients (focal loss)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss_sum += loss.item()

            # For metrics: use original (non-mixed) predictions
            train_calc.update(labels.cpu().numpy(),
                              F.softmax(preds, dim=1).detach().cpu().numpy())
            n_batches += 1

        train_loss = train_loss_sum / n_batches
        train_metrics = train_calc.compute_metrics()

        # ------ Validation phase (no MixUp, standard loss) ------
        model.eval()
        val_calc = MetricsCalculator(num_classes=num_classes)
        val_loss_sum, n_val = 0.0, 0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                preds = model(imgs)

                if criterion is not None:
                    loss = criterion(preds, labels)
                else:
                    loss = F.cross_entropy(preds, labels)

                val_loss_sum += loss.item()
                val_calc.update(labels.cpu().numpy(),
                                F.softmax(preds, dim=1).cpu().numpy())
                n_val += 1

        val_loss = val_loss_sum / n_val
        val_metrics = val_calc.compute_metrics()

        # ------ Store history ------
        history['loss']['train'].append(float(train_loss))
        history['loss']['val'].append(float(val_loss))
        for m in ['accuracy', 'precision', 'recall', 'f1_score',
                  'specificity', 'roc_auc', 'qwk']:
            if m in train_metrics:
                history[m]['train'].append(train_metrics[m])
                history[m]['val'].append(val_metrics[m])

        current_qwk = val_metrics.get('qwk', 0)

        # Print progress every 10 epochs + first epoch
        if (epoch + 1) % 10 == 0 or epoch == start_epoch:
            print(f"  Epoch {epoch+1:3d}/{epochs} | "
                  f"Loss: {train_loss:.4f}/{val_loss:.4f} | "
                  f"Acc: {train_metrics.get('accuracy',0):.4f}/{val_metrics.get('accuracy',0):.4f} | "
                  f"QWK: {train_metrics.get('qwk',0):.4f}/{current_qwk:.4f}")

        # ------ Best model (val_QWK max) ------
        if current_qwk > best_val_qwk:
            best_val_qwk = current_qwk
            best_epoch = epoch + 1
            torch.save({
                'epoch': best_epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_qwk': best_val_qwk,
                'val_loss': val_loss,
            }, str(experiment_dir / 'best_model.pth'))

        # ------ Resume checkpoint (every epoch) ------
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': None,
            'best_val_qwk': best_val_qwk,
            'best_epoch': best_epoch,
            'history': history,
        }, str(resume_path))

        # ------ Periodic checkpoint ------
        if (epoch + 1) % config.get('checkpoint_interval', 10) == 0:
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
            }, str(experiment_dir / f'checkpoint_epoch_{epoch+1:03d}.pth'))

    print(f"  \u2713 Training done. Best: Epoch {best_epoch} (QWK: {best_val_qwk:.4f})")

    # ---- Test evaluation with best model ----
    best_path = experiment_dir / 'best_model.pth'
    if best_path.exists():
        ckpt = torch.load(str(best_path), map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f"  Loaded best model (Epoch {ckpt['epoch']}, QWK: {ckpt['val_qwk']:.4f})")

    model.eval()
    test_calc = MetricsCalculator(num_classes=num_classes)
    test_loss_sum, n_test = 0.0, 0

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs)
            loss = F.cross_entropy(preds, labels)  # Always CE for test eval
            test_loss_sum += loss.item()
            test_calc.update(labels.cpu().numpy(),
                             F.softmax(preds, dim=1).cpu().numpy())
            n_test += 1

    test_loss = test_loss_sum / n_test
    test_metrics = test_calc.compute_metrics()
    cm = test_calc.get_confusion_matrix()
    roc_data = test_calc.get_roc_curves()

    print(f"  Test: Loss={test_loss:.4f} | Acc={test_metrics.get('accuracy',0):.4f} | "
          f"F1={test_metrics.get('f1_score',0):.4f} | QWK={test_metrics.get('qwk',0):.4f}")

    # ---- Save training log CSV ----
    log_data = {'epoch': list(range(1, len(history['loss']['train']) + 1))}
    for m in metric_names:
        log_data[f'train_{m}'] = history[m]['train']
        log_data[f'val_{m}'] = history[m]['val']
    pd.DataFrame(log_data).to_csv(str(experiment_dir / 'training_log.csv'), index=False)

    # ---- Save test metrics JSON ----
    def _convert(obj):
        if isinstance(obj, dict):
            return {k: _convert(v) for k, v in obj.items()}
        elif isinstance(obj, (list, tuple)):
            return [_convert(i) for i in obj]
        elif isinstance(obj, (np.integer, np.floating)):
            return obj.item()
        return obj

    metrics_output = {
        'best_epoch': best_epoch,
        'best_val_qwk': float(best_val_qwk),
        'test_loss': float(test_loss),
        **{f'test_{k}': float(v) for k, v in test_metrics.items()},
        'confusion_matrix': cm.tolist(),
    }
    with open(str(experiment_dir / 'metrics.json'), 'w') as f:
        json.dump(_convert(metrics_output), f, indent=4)

    # ---- Save metrics CSV ----
    metrics_row = {
        'best_epoch': best_epoch,
        'best_val_qwk': float(best_val_qwk),
        'test_loss': float(test_loss),
    }
    for k, v in test_metrics.items():
        metrics_row[f'test_{k}'] = float(v)
    pd.DataFrame([metrics_row]).to_csv(str(experiment_dir / 'metrics.csv'), index=False)

    # Clean up resume checkpoint
    if resume_path.exists():
        os.remove(str(resume_path))

    return history, test_metrics, test_loss, cm, roc_data, best_epoch, best_val_qwk


print("\u2713 Advanced training engine ready")

✓ Advanced training engine ready


## 16. Advanced Experiment Runner

Reuses existing `create_experiment_loaders`, `create_model`, and `save_experiment_plots`.  
Calls the advanced training engine with Focal Loss / MixUp support.

In [17]:
ADVANCED_RESULTS_CSV = Path('experiments/results_summary.csv')  # Same file as Part 1


def run_advanced_experiment(model_name, technique, experiment_config=None):
    """
    Run a single advanced experiment (Focal Loss / MixUp support).

    Args:
        model_name: 'cvits' or 'dual_expert'
        technique: key from ADVANCED_EXPERIMENTS dict
        experiment_config: override config (optional)

    Returns:
        dict with test results, or None if skipped
    """
    if experiment_config is None:
        experiment_config = ADVANCED_EXPERIMENTS[technique]

    exp_name = f"{model_name}/{technique}"
    experiment_dir = Path('experiments') / model_name / technique

    print(f"\n{'='*80}")
    print(f"EXPERIMENT: {exp_name} \u2014 {experiment_config['name']}")
    print(f"{'='*80}")

    # Skip if already completed
    metrics_file = experiment_dir / 'metrics.json'
    resume_file = experiment_dir / 'resume_checkpoint.pth'
    if metrics_file.exists() and not resume_file.exists():
        print(f"  \u26a0 Already completed. Loading existing results.")
        with open(str(metrics_file)) as f:
            existing = json.load(f)
        print(f"  Test QWK: {existing.get('test_qwk', 'N/A')}")
        return existing

    # Check CLAHE availability
    if experiment_config.get('preprocessing') == 'clahe' and not HAS_CV2:
        print("  \u2717 Skipped: OpenCV not available for CLAHE")
        return None

    # 1. Create data loaders (reuses existing function — handles rotation + CLAHE)
    print("  Creating data loaders...")
    train_loader, val_loader, test_loader = create_experiment_loaders(
        experiment_config, batch_size=BASE_CONFIG['batch_size'])

    # 2. Create model (reuses existing factory)
    print(f"  Building model: {model_name}")
    model = create_model(model_name, num_classes=BASE_CONFIG['num_classes'])
    model = model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=BASE_CONFIG['lr'],
        weight_decay=BASE_CONFIG['weight_decay']
    )

    # 3. Train with advanced engine
    print(f"  Training for {BASE_CONFIG['epochs']} epochs...")
    start_time = time.time()

    history, test_metrics, test_loss, cm, roc_data, best_epoch, best_qwk = \
        train_advanced_experiment(
            model, optimizer, train_loader, val_loader, test_loader,
            experiment_dir, BASE_CONFIG, experiment_config
        )

    elapsed = time.time() - start_time
    print(f"  Training time: {elapsed/60:.1f} minutes")

    # 4. Generate plots (reuses existing visualization)
    print("  Generating visualizations...")
    save_experiment_plots(
        history, cm, roc_data, experiment_dir,
        f"{model_name.upper()} \u2014 {experiment_config['name']}"
    )

    # 5. Append to unified results CSV
    result_row = {
        'model': model_name,
        'technique': technique,
        'best_epoch': best_epoch,
        'best_val_qwk': float(best_qwk),
        'test_loss': float(test_loss),
        'training_time_min': round(elapsed / 60, 1),
    }
    for k, v in test_metrics.items():
        result_row[f'test_{k}'] = float(v)

    ADVANCED_RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
    if ADVANCED_RESULTS_CSV.exists():
        df = pd.read_csv(str(ADVANCED_RESULTS_CSV))
        df = df[~((df['model'] == model_name) & (df['technique'] == technique))]
        df = pd.concat([df, pd.DataFrame([result_row])], ignore_index=True)
    else:
        df = pd.DataFrame([result_row])
    df.to_csv(str(ADVANCED_RESULTS_CSV), index=False)

    # 6. Cleanup GPU
    del model, optimizer
    torch.cuda.empty_cache()

    print(f"  \u2713 Experiment complete: {exp_name}")
    return result_row


def run_all_advanced_experiments(model_name):
    """Run all advanced technique experiments for a specific model."""
    print(f"\n{'#'*80}")
    print(f"  RUNNING ALL ADVANCED EXPERIMENTS FOR: {model_name.upper()}")
    print(f"{'#'*80}")

    results = {}
    for technique in ADVANCED_EXPERIMENTS:
        result = run_advanced_experiment(model_name, technique)
        if result:
            results[technique] = result

    print(f"\n\u2713 All advanced {model_name} experiments complete "
          f"({len(results)}/{len(ADVANCED_EXPERIMENTS)} succeeded)")
    return results


print("\u2713 Advanced experiment runner ready")

✓ Advanced experiment runner ready


## 17. Run All Advanced CViTS-Net Experiments

Execute Rotation+Focal Loss, Rotation+MixUp, CLAHE+Rotation+Focal Loss with CViTS-Net.

In [18]:
adv_cvits_results = run_all_advanced_experiments('cvits')


################################################################################
  RUNNING ALL ADVANCED EXPERIMENTS FOR: CVITS
################################################################################

EXPERIMENT: cvits/rotation_focal_loss — Rotation + Focal Loss
  Creating data loaders...
  Building model: cvits
  Model: cvits | Parameters: 30,338,053
  Training for 100 epochs...
  Loss: Focal Loss (gamma=2.0)
  Epoch   1/100 | Loss: 13.1388/13.4492 | Acc: 0.2769/0.2733 | QWK: 0.0050/0.0000
  Epoch  10/100 | Loss: 13.3733/13.4492 | Acc: 0.2727/0.2733 | QWK: 0.0000/0.0000
  Epoch  20/100 | Loss: 13.4267/13.4492 | Acc: 0.2727/0.2733 | QWK: 0.0000/0.0000
  Epoch  30/100 | Loss: 13.4267/13.4492 | Acc: 0.2727/0.2733 | QWK: 0.0000/0.0000
  Epoch  40/100 | Loss: 13.4267/13.4492 | Acc: 0.2727/0.2733 | QWK: 0.0000/0.0000
  Epoch  50/100 | Loss: 13.3733/13.4492 | Acc: 0.2727/0.2733 | QWK: 0.0000/0.0000
  Epoch  60/100 | Loss: 13.4267/13.4492 | Acc: 0.2727/0.2733 | QWK: 0.0000/0.0000
  E

KeyboardInterrupt: 

## 18. Run All Advanced Dual-Expert Experiments

Execute Rotation+Focal Loss, Rotation+MixUp, CLAHE+Rotation+Focal Loss with Dual-Expert.

In [ ]:
adv_dual_expert_results = run_all_advanced_experiments('dual_expert')

## 19. Final Comparison — All Experiments

Comprehensive comparison table covering **all 12 experiments** (6 from Part 1 + 6 from Part 2).

In [ ]:
print("\n" + "="*80)
print("FINAL COMPARISON — ALL EXPERIMENTS (Part 1 + Part 2)")
print("="*80)

results_path = Path('experiments/results_summary.csv')

if results_path.exists():
    df_all = pd.read_csv(str(results_path))

    # ---- Full metrics table ----
    display_cols = ['model', 'technique', 'best_epoch',
                    'test_accuracy', 'test_precision', 'test_recall',
                    'test_f1_score', 'test_specificity', 'test_roc_auc',
                    'test_qwk', 'test_loss', 'training_time_min']
    cols = [c for c in display_cols if c in df_all.columns]
    print("\n" + df_all[cols].to_string(index=False))

    # ---- Key comparison table (Accuracy, F1, ROC-AUC, QWK) ----
    print("\n" + "="*80)
    print("KEY METRICS COMPARISON")
    print("="*80)
    key_cols = ['model', 'technique', 'test_accuracy', 'test_f1_score',
                'test_roc_auc', 'test_qwk']
    key_cols = [c for c in key_cols if c in df_all.columns]
    print("\n" + df_all[key_cols].to_string(index=False))

    # ---- Best technique per model ----
    print("\n" + "-"*60)
    for model_name in df_all['model'].unique():
        model_df = df_all[df_all['model'] == model_name]
        if 'test_qwk' in model_df.columns and not model_df['test_qwk'].isna().all():
            best = model_df.loc[model_df['test_qwk'].idxmax()]
            print(f"  Best for {model_name}: {best['technique']} "
                  f"(QWK={best['test_qwk']:.4f}, Acc={best.get('test_accuracy', 0):.4f}, "
                  f"F1={best.get('test_f1_score', 0):.4f})")

    # ---- Overall best ----
    if 'test_qwk' in df_all.columns:
        overall_best = df_all.loc[df_all['test_qwk'].idxmax()]
        print(f"\n  \u2b50 Overall Best: {overall_best['model']}/{overall_best['technique']} "
              f"(QWK={overall_best['test_qwk']:.4f})")

    print(f"\n\u2713 Full results saved: {results_path}")
    print(f"   Total experiments: {len(df_all)}")
else:
    print("No results found. Run experiments first.")

print("\n" + "="*80)
print("\u2705 All experiments complete!")
print("="*80)